In [1]:
import tensorflow as tf
import time

In [2]:
tf.__version__

'2.20.0'

In [14]:
class FileDataset(tf.data.Dataset):
    def read_files_in_batches(num_samples):
        # open file
        time.sleep(0.03)
        for sample_idx in range(num_samples):
            time.sleep(0.015)
            yield (sample_idx,)
            
    def __new__(cls, num_samples=10):
        print("new called")
        return tf.data.Dataset.from_generator(
            cls.read_files_in_batches,
            output_signature=tf.TensorSpec(shape=(1,), dtype=tf.int64),
            args=(num_samples,)
        )

In [15]:
def benchmark(dataset, num_epochs=2):
    for epochs_num in range(num_epochs):
        for sample in dataset:
            time.sleep(0.01)

In [18]:
%%timeit
benchmark(FileDataset())

new called
new called
new called
new called
new called
new called
new called
new called
558 ms ± 43 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit
benchmark(FileDataset().prefetch(1))

new called
new called
new called
new called
new called
new called
new called
new called
518 ms ± 11.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
%%timeit
benchmark(FileDataset().prefetch(tf.data.AUTOTUNE))

new called
new called
new called
new called
new called
new called
new called
new called
599 ms ± 62.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
dataset = tf.data.Dataset.range(5)
for d in dataset:
    print(d.numpy())

0
1
2
3
4


In [27]:
dataset = dataset.map(lambda x: x**2)
for d in dataset:
    print(d.numpy())

0
1
4
9
16


In [30]:
dataset = dataset.cache()
for d in dataset.as_numpy_iterator():
    print(d)

0
1
4
9
16


In [31]:
list(dataset.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]

In [32]:
list(dataset.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]

In [37]:
def mapped_function(s):
    tf.py_function(lambda: time.sleep(0.03),[], ())
    return s

In [41]:
%%timeit -n1 -r1

benchmark(FileDataset().map(mapped_function), 5)

new called
2.91 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [42]:
%%timeit -n1 -r1

benchmark(FileDataset().map(mapped_function).cache(), 5)

new called
1.19 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
